# sfmc_sync_re_email_status
Syncs SFMC opt-out / bounce flags to Blackbaud SKY API.

For each constituent whose email is in the SFMC opt-out or bounce list:
1. GET existing comm preferences — skip if "Do Not Email" already set
2. POST "Do Not Email" solicit code
3. POST audit action
4. POST action custom field

Set `DRY_RUN = True` to preview without making API calls.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb
import pandas as pd


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RE_EMAIL_ADDRESSES_DATASET = "raw_re_email_addresses"
RE_CONSTITUENTS_DATASET    = "bronze_re_constituents"
SFMC_BOUNCE_EVENTS_DATASET = "085c865f-2a21-4f04-9333-8e5c24789a6b"
SFMC_OPTOUTS_DATASET       = "OUT | All SFMC opt outs (wide)"

COMM_PREF_URL  = f"{API_BASE}/constituent/v1/communicationpreferences"
ACTIONS_URL    = f"{API_BASE}/constituent/v1/actions"
ACTION_CF_URL  = f"{API_BASE}/constituent/v1/actions/customfields"
AUTHOR_EMAIL   = "data_insights@heartfoundation.org.au"

DRY_RUN = True   # set False to send actual API calls
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
print("Loading datasets...")
raw_email_df     = domo.read_dataframe(RE_EMAIL_ADDRESSES_DATASET,  query="SELECT * FROM table")
constituents_df  = domo.read_dataframe(RE_CONSTITUENTS_DATASET,     query="SELECT * FROM table")
bounce_events_df = domo.read_dataframe(SFMC_BOUNCE_EVENTS_DATASET,  query="SELECT * FROM table")
optouts_df       = domo.read_dataframe(SFMC_OPTOUTS_DATASET,        query="SELECT * FROM table")
print(f"  raw_email_df     : {len(raw_email_df):,}")
print(f"  constituents_df  : {len(constituents_df):,}")
print(f"  bounce_events_df : {len(bounce_events_df):,}")
print(f"  optouts_df       : {len(optouts_df):,}")


In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def normalize_email(s):
    if isinstance(s, pd.Series):
        return s.astype("string").str.strip().str.lower().fillna("")
    return str(s).strip().lower() if s and not pd.isna(s) else ""

def to_bool(v) -> bool:
    if isinstance(v, bool): return v
    if isinstance(v, str):  return v.strip().upper() in ("TRUE","Y","YES","1")
    return bool(v) if v is not None else False

def get_unsubscribe_emails(df: pd.DataFrame) -> set:
    cols      = [c for c in ["Revenue_optout","Manual_optout","Global_optout"] if c in df.columns]
    email_col = next((c for c in ["Email Address","Email","email"] if c in df.columns), None)
    if not cols or not email_col: return set()
    mask = df[cols].apply(lambda c: c.astype(str).str.upper() == "Y").any(axis=1)
    return set(normalize_email(df.loc[mask, email_col]))

def get_inactive_emails(df: pd.DataFrame) -> set:
    col       = next((c for c in ["Held_and_hard_bounce_optout"] if c in df.columns), None)
    email_col = next((c for c in ["Email Address","Email","email"] if c in df.columns), None)
    if not col or not email_col: return set()
    mask = df[col].astype(str).str.upper() == "Y"
    return set(normalize_email(df.loc[mask, email_col]))

def bounce_email_set(df: pd.DataFrame, event_type: str) -> set:
    if "EventType" not in df.columns: return set()
    key_col = next((c for c in ["SubscriberKey","EmailAddress","email"] if c in df.columns), None)
    if not key_col: return set()
    return set(normalize_email(df.loc[df["EventType"] == event_type, key_col]))


In [ ]:
# ── Build work queue ──────────────────────────────────────────────────────────
unsubscribe_emails = get_unsubscribe_emails(optouts_df)
inactive_emails    = get_inactive_emails(optouts_df) | bounce_email_set(bounce_events_df, "HardBounce")

print(f"Emails to unsubscribe  : {len(unsubscribe_emails):,}")
print(f"Emails to mark inactive: {len(inactive_emails):,}")

if "address" in raw_email_df.columns:
    raw_email_df["email_address"] = normalize_email(raw_email_df["address"])
    post_queue_df = (
        raw_email_df[raw_email_df["email_address"].isin(unsubscribe_emails)]
        [["email_address", "constituent_id"]]
        .dropna(subset=["constituent_id"])
        .assign(constituent_id=lambda d: d["constituent_id"].astype(str).str.strip())
        .drop_duplicates(subset=["constituent_id"])
        .reset_index(drop=True)
    )
else:
    post_queue_df = pd.DataFrame(columns=["email_address", "constituent_id"])

print(f"\nConstituents in post queue: {len(post_queue_df):,}")


In [ ]:
# ── Step 1: GET existing comm preferences — skip already-set constituents ──────
# Prevents duplicate solicit codes and avoids unnecessary API writes.
# Uses api_request_with_auth so the token auto-refreshes if it expires mid-run.

if DRY_RUN:
    print("DRY_RUN=True — skipping GET check, showing queue only:")
    print(post_queue_df.head(10).to_string(index=False))
    filtered_queue_df = post_queue_df.copy()
else:
    token_mgr = TokenManager(interactive=False)
    sess      = requests.Session()
    check_results = []

    print(f"Checking existing comm preferences for {len(post_queue_df):,} constituents...")

    for _, row in post_queue_df.iterrows():
        cid  = row["constituent_id"]
        resp = api_request_with_auth(
            "GET", f"{COMM_PREF_URL}?constituent_id={cid}",
            token_mgr=token_mgr, session=sess,
        )
        if resp.ok:
            data  = resp.json()
            prefs = data if isinstance(data, list) else data.get("value", [])
            names = [p.get("solicit_code", "") for p in prefs if isinstance(p, dict)]
            has_dne = any("do not email" in n.lower() for n in names)
        else:
            has_dne = False

        check_results.append({
            "constituent_id":  cid,
            "email_address":   row["email_address"],
            "has_do_not_email": has_dne,
            "check_status":    resp.status_code,
        })

    check_df    = pd.DataFrame(check_results)
    already_set = check_df["has_do_not_email"].sum()
    print(f"  Already have Do Not Email: {already_set:,}")
    print(f"  Need to be updated       : {len(check_df) - already_set:,}")
    filtered_queue_df = check_df[~check_df["has_do_not_email"]].copy()


In [ ]:
# ── Steps 2-4: POST comm preference → POST action → POST action custom field ──
# token_mgr handles auto-refresh on 401 so long-running batches stay authenticated.

if DRY_RUN:
    print("DRY_RUN=True — no POST calls sent.")
    print(f"Would process {len(filtered_queue_df):,} constituents.")
else:
    from datetime import datetime as _dt
    today     = _dt.now().strftime("%Y-%m-%dT00:00:00Z")
    token_mgr = TokenManager(interactive=False)
    sess      = requests.Session()
    results   = []

    for i, (_, row) in enumerate(filtered_queue_df.iterrows(), 1):
        cid    = row["constituent_id"]
        result = {
            "constituent_id": cid,
            "email_address":  row["email_address"],
            "cp_status":      None,
            "action_status":  None,
            "action_id":      None,
            "cf_status":      None,
        }

        # ── Step 2: POST comm preference ──────────────────────────────────
        cp_resp = api_request_with_auth(
            "POST", COMM_PREF_URL, token_mgr=token_mgr, session=sess,
            json_body={"constituent_id": cid, "solicit_code": "Do Not Email",
                       "start": None, "end": None},
        )
        result["cp_status"] = cp_resp.status_code

        if not cp_resp.ok:
            print(f"  [{i}] {cid} comm pref failed: HTTP {cp_resp.status_code} {cp_resp.text[:200]}")
            results.append(result)
            continue

        # ── Step 3: POST action ────────────────────────────────────────────
        act_resp = api_request_with_auth(
            "POST", ACTIONS_URL, token_mgr=token_mgr, session=sess,
            json_body={
                "constituent_id": cid,
                "author":         AUTHOR_EMAIL,
                "category":       "Task/Other",
                "completed":      True,
                "completed_date": today,
                "date":           today,
                "description":    "Do Not Email added as opted out in SFMC",
                "direction":      "Outbound",
                "outcome":        "Successful",
                "priority":       "Normal",
                "status":         "Completed",
                "summary":        "Solicit code added",
                "type":           "Solicit Code Change",
            },
        )
        result["action_status"] = act_resp.status_code
        action_id = act_resp.json().get("id", "") if act_resp.ok else ""
        result["action_id"] = action_id

        # ── Step 4: POST action custom field ──────────────────────────────
        if action_id:
            cf_resp = api_request_with_auth(
                "POST", ACTION_CF_URL, token_mgr=token_mgr, session=sess,
                json_body={
                    "category":  "Solicit Code Added",
                    "comment":   "SFMC",
                    "date":      today,
                    "parent_id": str(action_id),
                    "value":     "Do Not Email",
                },
            )
            result["cf_status"] = cf_resp.status_code

        results.append(result)

        if i % 50 == 0:
            print(f"  Progress: {i:,}/{len(filtered_queue_df):,}")

    results_df = pd.DataFrame(results)
    cp_ok  = (results_df["cp_status"].astype(str).str[:1] == "2").sum()
    act_ok = (results_df["action_status"].astype(str).str[:1] == "2").sum()
    cf_ok  = (results_df["cf_status"].astype(str).str[:1] == "2").sum()

    print(f"\n{'='*50}")
    print(f"Comm prefs posted    : {cp_ok:,} / {len(results_df):,}")
    print(f"Actions posted       : {act_ok:,} / {len(results_df):,}")
    print(f"Action custom fields : {cf_ok:,} / {len(results_df):,}")
    print(f"{'='*50}")
    print(results_df.head(10).to_string(index=False))
